In [30]:
import requests
import polars as pl
import json
from datetime import datetime

## GET ALL THE DATA FROM API | approx 950k json files

In [ ]:
from pathlib import Path
# all prices 
all_prices_url = "https://prices.azure.com/api/retail/prices?currencyCode='USD'"
all_output_dir = Path("putput_data/all_USD_json")
overwrite_existing = False

all_output_dir.mkdir(exist_ok=True)

existing_json_files = list(all_output_dir.glob("*.json"))
if existing_json_files and not overwrite_existing:
    raise FileExistsError(
        f"Folder {all_output_dir} contains {len(existing_json_files)} JSON files. "
        "Set overwrite_existing = True to replace them."
    )

if overwrite_existing:
    for existing_file in existing_json_files:
        existing_file.unlink()

url = all_prices_url
page = 1
total_records = 0

while url:
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    payload = response.json()
    items = payload.get("Items", [])

    for item_index, item in enumerate(items, start=1):
        total_records += 1
        item_file = all_output_dir / f"item_{total_records:09d}_page_{page:06d}_row_{item_index:04d}.json"
        with item_file.open("w", encoding="utf-8") as f:
            json.dump(item, f, indent=2, ensure_ascii=False)

    print(f"Page {page}: saved {len(items)} items; total {total_records}")

    url = payload.get("NextPageLink")
    page += 1

print(f"Saved {total_records} price items to {all_output_dir.resolve()}")

## SAVE TO PARQUET

In [31]:
from pathlib import Path

items_dir = Path("output_data/all_USD_json")
files = sorted(items_dir.glob("*.json"))

records = []
for file in files:
    with file.open(encoding="utf-8") as f:
        record = json.load(f)
    record["source_file"] = file.name
    records.append(record)

df_items = pl.DataFrame(records, infer_schema_length=None)
df_items.write_parquet('output_data/all_USD.parquet')

# GET armRegionName, productName, serviceName - distinct lists

In [33]:
all_USD = pl.read_parquet('output_data/all_USD.parquet')

arm_region_names = (
    all_USD
    .select("armRegionName")
    .unique()
    .sort("armRegionName")
    .to_series()
    .to_list()
)

product_names = (
    all_USD
    .select("productName")
    .unique()
    .sort("productName")
    .to_series()
    .to_list()
)

service_names = (
    all_USD
    .select("serviceName")
    .unique()
    .sort("serviceName")
    .to_series()
    .to_list()
)

In [34]:
preferred_columns = [
    "currencyCode",
    "armRegionName",
    "location",
    "type",
    "serviceFamily",
    "serviceName",
    "productName",
    "skuName",
    "armSkuName",
    "meterName",
    "unitOfMeasure",
    "retailPrice",
    "unitPrice",
]

remaining_columns = [
    column
    for column in all_USD.columns
    if column not in preferred_columns
]

all_USD_sorted = all_USD.select(preferred_columns + remaining_columns)

In [36]:
all_USD_sorted.write_excel(
    workbook="output_data/all_USD.xlsx",
    worksheet="items",
    autofit=True,
    freeze_panes="A2",
)

# GET REST API DATA WITH FILTERS

In [ ]:
params = {
    "currencyCode": ["USD"],
    "location": ["US East"],
    "type": ["Consumption"],
    "serviceName": ["Azure Cognitive Search"],
    "productName": ["Azure AI Search"],
}

clean_url = "https://prices.azure.com/api/retail/prices"

timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
file_name = f"costs_summary_{timestamp}"
excel_file_name = f"{file_name}.xlsx"

def filter_values(value):
    if value in (None, ""):
        return []
    if isinstance(value, (list, tuple, set)):
        return [item for item in value if item not in (None, "")]
    return [value]


def escape_odata_value(value):
    return str(value).replace("'", "''")


filters = []
for key, value in params.items():
    value_filters = [f"{key} eq '{escape_odata_value(item)}'" for item in filter_values(value)]
    if len(value_filters) == 1:
        filters.append(value_filters[0])
    elif len(value_filters) > 1:
        filters.append(f"({' or '.join(value_filters)})")

new_url = clean_url + '?$filter=' + ' and '.join(filters)
print(new_url)

response_text = requests.get(new_url).text

https://prices.azure.com/api/retail/prices?$filter=currencyCode eq 'USD' and location eq 'US East' and type eq 'Consumption' and serviceName eq 'Azure Cognitive Search' and productName eq 'Azure AI Search'


In [38]:
data = json.loads(response_text)

df_response = pl.DataFrame(data["Items"])

preferred_columns = [
    "currencyCode",
    "armRegionName",
    "location",
    "type",
    "serviceFamily",
    "serviceName",
    "productName",
    "skuName",
    "armSkuName",
    "meterName",
    "unitOfMeasure",
    "retailPrice",
    "unitPrice",
]

remaining_columns = [
    column
    for column in df_response.columns
    if column not in preferred_columns
]

df_response = df_response.select(preferred_columns + remaining_columns)

df_response.write_excel(
    workbook="output_data/filtered_USD.xlsx",
    worksheet="items",
    autofit=True,
    freeze_panes="A2",
)